In [2]:
# ============================================================
# IMPORTS E CONFIGURAÇÃO
# ------------------------------------------------------------
# Projeto: Risco de abandono escolar em Roraima
# Fase 1 : smoke test — 2025 apenas, base nova (recorte RR)
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# sklearn — pré-processamento
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# sklearn — modelo e avaliação
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.ensemble import (
    RandomForestRegressor, HistGradientBoostingRegressor,
)
from sklearn.linear_model import Ridge

# visualização
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURAÇÃO
# ============================================================
RANDOM_STATE = 1
np.random.seed(RANDOM_STATE)

# Silenciar warnings comuns (pandas futuro, sklearn, openpyxl)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# ============================================================
# CAMINHOS
# ------------------------------------------------------------
# O notebook roda em notebooks/. A raiz do repositório é um
# nível acima. Detecta isso automaticamente para funcionar
# também se for executado da raiz por engano.
# ============================================================
CWD = Path.cwd()
BASE = CWD.parent if CWD.name == 'notebooks' else CWD

DADOS        = BASE / 'dados'
INTERIM      = DADOS / 'interim' / 'indicadores_rr'
PROCESSADO   = DADOS / 'processado'

# Produto final do bootstrap
BASE_LONG = PROCESSADO / 'base_longitudinal_rr_2019_2025.parquet'

print('CWD        :', CWD)
print('BASE       :', BASE)
print('INTERIM    :', INTERIM, '→', INTERIM.exists())
print('PROCESSADO :', PROCESSADO, '→', PROCESSADO.exists())
print('Base long. :', BASE_LONG, '→', BASE_LONG.exists())

CWD        : c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\notebooks
BASE       : c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo
INTERIM    : c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\dados\interim\indicadores_rr → True
PROCESSADO : c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\dados\processado → True
Base long. : c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\dados\processado\base_longitudinal_rr_2019_2025.parquet → True


In [3]:
df = pd.read_parquet(BASE_LONG)
print(df.shape)
print(df.dtypes.value_counts())
df.head(3)

(6054, 59)
Float64           46
string[python]     6
Int8               5
Int16              2
Name: count, dtype: int64


,ANO,ANO_ALVO,CO_ENTIDADE,NO_ENTIDADE,CO_MUNICIPIO,NO_MUNICIPIO,LOCALIZACAO,DEPENDENCIA,REDE_PUBLICA,DISP_ATU,...,ABANDONO_MED_LAG1,TDI_FUN_AF_DELTA1,TDI_MED_TOTAL_DELTA1,ATU_FUN_AF_DELTA1,ATU_MED_TOTAL_DELTA1,ABANDONO_FUN_AF_DELTA1,ABANDONO_MED_DELTA1,ABANDONO_FUN_T1,ABANDONO_FUN_AF_T1,ABANDONO_MED_T1
0,2019,2020,14000016,COLEGIO ESTADUAL MILITARIZADO DESEMBARGADOR SA...,1400050,Alto Alegre,Urbana,Estadual,1,1,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0
1,2019,2020,14000032,ESCOLA ESTADUAL INDIGENA ANTONIO DIAS DE SOUZA...,1400050,Alto Alegre,Rural,Estadual,1,1,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0
2,2019,2020,14000113,ESCOLA ESTADUAL INDIGENA FABIO DE MAGALHAES,1400050,Alto Alegre,Rural,Estadual,1,1,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,<NA>,<NA>


In [ ]:
# Todas as colunas, com tipo e % de nulos
resumo = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'n_nulos': df.isna().sum(),
    'pct_nulos': (df.isna().mean() * 100).round(1),
})
print(f'Total de colunas: {len(df.columns)}')
print(resumo.to_string())

Total de colunas: 59
                           tipo  n_nulos  pct_nulos
ANO                       Int16        0        0.0
ANO_ALVO                  Int16      960       15.9
CO_ENTIDADE              string        0        0.0
NO_ENTIDADE              string        0        0.0
CO_MUNICIPIO             string        0        0.0
NO_MUNICIPIO             string        0        0.0
LOCALIZACAO              string        0        0.0
DEPENDENCIA              string        0        0.0
REDE_PUBLICA               Int8        0        0.0
DISP_ATU                   Int8        0        0.0
DISP_TDI                   Int8        0        0.0
DISP_IED                   Int8        0        0.0
DISP_REND                  Int8        0        0.0
ATU_FUN_TOTAL           Float64     1473       24.3
ATU_FUN_AI              Float64     4177       69.0
ATU_FUN_AF              Float64     4316       71.3
ATU_MED_TOTAL           Float64     4836       79.9
TDI_FUN_TOTAL           Float64     1473   

In [5]:
alvos = ['ABANDONO_FUN_T1', 'ABANDONO_FUN_AF_T1', 'ABANDONO_MED_T1']

print('Distribuição por ANO_ALVO:')
print(df.groupby('ANO_ALVO')[alvos].agg(['count', 'mean', 'std']).round(3))

print('\nEstatísticas dos alvos (todos os anos juntos):')
print(df[alvos].describe().round(3))

Distribuição por ANO_ALVO:
         ABANDONO_FUN_T1               ABANDONO_FUN_AF_T1                \
                   count   mean    std              count   mean    std   
ANO_ALVO                                                                  
2020                 635  0.681  2.407                249  0.992  2.604   
2021                 616  1.612  3.984                246  2.958  4.731   
2022                 633   2.23  4.695                248  4.943  5.955   
2023                 658  1.464  3.853                244    3.0  4.451   
2024                 664   1.18  2.914                251  2.628  3.717   
2025                 664  0.984  2.639                256  2.415  4.131   

         ABANDONO_MED_T1                  
                   count    mean     std  
ANO_ALVO                                  
2020                 167   1.598   4.374  
2021                 167   5.541   8.255  
2022                 171  11.411  10.118  
2023                 176   7.184   7.70

In [ ]:
# Para o smoke test, um ano só: ANO=2024 → ANO_ALVO=2025
df_2025 = df[df['ANO_ALVO'] == 2025].copy()

print(f'Linhas: {len(df_2025)}')
print(f'Escolas únicas: {df_2025["CO_ENTIDADE"].nunique()}')
print(f'\nValores não-nulos por alvo:')
for a in alvos:
    print(f'  {a}: {df_2025[a].notna().sum()}')

Linhas: 885
Escolas únicas: 885

Valores não-nulos por alvo:
  ABANDONO_FUN_T1: 664
  ABANDONO_FUN_AF_T1: 256
  ABANDONO_MED_T1: 181


In [13]:
# ============================================================
# SMOKE TEST — 2025, anos finais, escolas públicas
# ============================================================
TARGET = 'ABANDONO_FUN_AF_T1'

# Filtra: ano alvo 2025 + rede pública + tem alvo não-nulo
df_smoke = df[
    (df['ANO_ALVO'] == 2025) &
    (df['REDE_PUBLICA'] == 1) &
    (df[TARGET].notna())
].copy()

print(f'Escolas (linhas): {len(df_smoke)}')
print(f'Colunas         : {df_smoke.shape[1]}')
print(f'\nDistribuição do alvo ({TARGET}):')
print(df_smoke[TARGET].describe().round(3))

Escolas (linhas): 243
Colunas         : 59

Distribuição do alvo (ABANDONO_FUN_AF_T1):
count    243.0
mean     2.542
std      4.202
min        0.0
25%        0.0
50%        0.8
75%       3.05
max       25.0
Name: ABANDONO_FUN_AF_T1, dtype: Float64


In [14]:
# Colunas que NÃO entram como feature
DROP = [
    # IDs e metadados
    'CO_ENTIDADE', 'NO_ENTIDADE',
    'NO_MUNICIPIO',
    'ANO', 'ANO_ALVO',
    # Alvos irmãos (leakage direto: mesmo período do alvo)
    'ABANDONO_FUN_T1', 'ABANDONO_MED_T1',
    # O próprio alvo
    TARGET,
]

# CO_MUNICIPIO fica como categórica (distingue capital/interior)
X = df_smoke.drop(columns=[c for c in DROP if c in df_smoke.columns])
y = df_smoke[TARGET].copy()

print(f'X: {X.shape}   y: {y.shape}')

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include=['string', 'object']).columns.tolist()

print(f'\nNuméricas  ({len(num_cols)}): {num_cols}')
print(f'Categóricas ({len(cat_cols)}): {cat_cols}')

X: (243, 51)   y: (243,)

Numéricas  (48): ['REDE_PUBLICA', 'DISP_ATU', 'DISP_TDI', 'DISP_IED', 'DISP_REND', 'ATU_FUN_TOTAL', 'ATU_FUN_AI', 'ATU_FUN_AF', 'ATU_MED_TOTAL', 'TDI_FUN_TOTAL', 'TDI_FUN_AI', 'TDI_FUN_AF', 'TDI_MED_TOTAL', 'IED_FUN_N1', 'IED_FUN_N2', 'IED_FUN_N3', 'IED_FUN_N4', 'IED_FUN_N5', 'IED_FUN_N6', 'IED_FUN_ALTO', 'IED_MED_N1', 'IED_MED_N2', 'IED_MED_N3', 'IED_MED_N4', 'IED_MED_N5', 'IED_MED_N6', 'IED_MED_ALTO', 'APROVACAO_FUN', 'APROVACAO_FUN_AI', 'APROVACAO_FUN_AF', 'APROVACAO_MED', 'REPROVACAO_FUN', 'REPROVACAO_FUN_AI', 'REPROVACAO_FUN_AF', 'REPROVACAO_MED', 'ABANDONO_FUN', 'ABANDONO_FUN_AI', 'ABANDONO_FUN_AF', 'ABANDONO_MED', 'ABANDONO_FUN_LAG1', 'ABANDONO_FUN_AF_LAG1', 'ABANDONO_MED_LAG1', 'TDI_FUN_AF_DELTA1', 'TDI_MED_TOTAL_DELTA1', 'ATU_FUN_AF_DELTA1', 'ATU_MED_TOTAL_DELTA1', 'ABANDONO_FUN_AF_DELTA1', 'ABANDONO_MED_DELTA1']
Categóricas (3): ['CO_MUNICIPIO', 'LOCALIZACAO', 'DEPENDENCIA']


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Split aleatório (só 2025, não dá para fazer temporal)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')

# Preprocessor
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

modelos = {
    'Ridge'                : Ridge(alpha=1.0),
    'RandomForest'         : RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'HistGradientBoosting' : HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.05, max_depth=6,
        random_state=RANDOM_STATE
    ),
}

resultados = {}
for nome, modelo in modelos.items():
    pipe = Pipeline([('pre', preprocessor), ('model', modelo)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    resultados[nome] = {
        'MAE' : mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R²'  : r2_score(y_test, pred),
    }

import pandas as pd
print('\n=== Resultados (2025, anos finais, escolas públicas) ===')
print(pd.DataFrame(resultados).T.round(3))

Treino: (194, 51) | Teste: (49, 51)

=== Resultados (2025, anos finais, escolas públicas) ===
                        MAE   RMSE     R²
Ridge                 2.604  4.085  0.204
RandomForest          2.490  3.979  0.245
HistGradientBoosting  2.442  3.824  0.303


In [16]:
# ============================================================
# EDA COMPLETA — RELATÓRIO ESTRUTURADO PARA IA
# ------------------------------------------------------------
# Objetivo: extrair TUDO que um agente precisa saber sobre a
# base antes de decidir o pré-processamento e a modelagem.
# ============================================================
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 250)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# Recalcula BASE_LONG se não estiver definido
if 'BASE_LONG' not in dir():
    _cwd = Path.cwd()
    _base = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    BASE_LONG = _base / 'dados' / 'processado' / 'base_longitudinal_rr_2019_2025.parquet'

df = pd.read_parquet(BASE_LONG)

ALVOS      = ['ABANDONO_FUN_T1', 'ABANDONO_FUN_AF_T1', 'ABANDONO_MED_T1']
META       = ['ANO', 'ANO_ALVO']
IDS        = ['CO_ENTIDADE', 'NO_ENTIDADE', 'CO_MUNICIPIO', 'NO_MUNICIPIO']
CATEGORICAS_ESPERADAS = ['CO_MUNICIPIO', 'LOCALIZACAO', 'DEPENDENCIA']

def secao(n, titulo):
    print('\n' + '=' * 78)
    print(f'  {n}. {titulo}')
    print('=' * 78)

def sub(t):
    print(f'\n--- {t} ---')

# ============================================================
# 1. FORMA E TIPOS
# ============================================================
secao(1, 'FORMA E TIPOS')
print(f'Linhas  : {len(df):,}')
print(f'Colunas : {df.shape[1]}')
print('\nDistribuição de dtypes:')
print(df.dtypes.value_counts().to_string())

num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include=['string', 'object']).columns.tolist()

sub(f'Colunas numéricas ({len(num_cols)})')
print(num_cols)
sub(f'Colunas categóricas/texto ({len(cat_cols)})')
print(cat_cols)

# ============================================================
# 2. COBERTURA TEMPORAL
# ============================================================
secao(2, 'COBERTURA TEMPORAL')
print('Anos presentes em ANO      :', sorted(df['ANO'].dropna().unique().tolist()))
print('Anos presentes em ANO_ALVO :', sorted(df['ANO_ALVO'].dropna().unique().tolist()))

sub('Pares (ANO → ANO_ALVO) e tamanho')
pares = df.groupby(['ANO', 'ANO_ALVO'], dropna=False).size().rename('n').reset_index()
print(pares.to_string(index=False))

sub('Escolas únicas por ANO_ALVO')
print(df.groupby('ANO_ALVO')['CO_ENTIDADE'].nunique().to_string())

# ============================================================
# 3. VALORES AUSENTES
# ============================================================
secao(3, 'VALORES AUSENTES')
nulos = pd.DataFrame({
    'n_nulos' : df.isna().sum(),
    'pct_nulos': (df.isna().mean() * 100).round(1),
}).sort_values('pct_nulos', ascending=False)
print(nulos.to_string())

# ============================================================
# 4. COLUNAS CONSTANTES OU QUASE CONSTANTES
# ============================================================
secao(4, 'COLUNAS CONSTANTES OU QUASE CONSTANTES')
constantes = []
quase = []
for c in df.columns:
    n_unique = df[c].nunique(dropna=True)
    if n_unique <= 1:
        constantes.append(c)
    elif n_unique <= 3 and df[c].dtype.kind in 'ifb':
        quase.append((c, n_unique, df[c].value_counts(dropna=False).head(3).to_dict()))
print(f'Totalmente constantes: {constantes}')
print(f'\nQuase constantes (≤3 valores):')
for c, n, vals in quase:
    print(f'  {c}: {n} únicos → {vals}')

# ============================================================
# 5. CARDINALIDADE DE CATEGÓRICAS
# ============================================================
secao(5, 'CARDINALIDADE DAS CATEGÓRICAS')
for c in cat_cols:
    n_unique = df[c].nunique(dropna=True)
    print(f'\n{c}: {n_unique} únicos')
    vc = df[c].value_counts(dropna=False).head(10)
    print(vc.to_string())

# ============================================================
# 6. ESTATÍSTICAS DESCRITIVAS DAS NUMÉRICAS
# ============================================================
secao(6, 'ESTATÍSTICAS DAS NUMÉRICAS')
desc = df[num_cols].describe().T
desc['nulos'] = df[num_cols].isna().sum().values
desc['n_unicos'] = df[num_cols].nunique().values
print(desc.round(4).to_string())

# ============================================================
# 7. ANÁLISE DOS ALVOS
# ============================================================
secao(7, 'ANÁLISE DOS ALVOS')
for alvo in ALVOS:
    sub(f'Alvo: {alvo}')
    print(f'Não nulos: {df[alvo].notna().sum():,} ({df[alvo].notna().mean()*100:.1f}%)')
    print('Por ANO_ALVO:')
    tab = df.groupby('ANO_ALVO')[alvo].agg(
        n='count', media='mean', mediana='median', std='std',
        p75=lambda s: s.quantile(0.75), max='max'
    ).round(3)
    print(tab.to_string())
    print(f'Zeros no alvo: {(df[alvo] == 0).sum():,} ({(df[alvo] == 0).mean()*100:.1f}%)')

# ============================================================
# 8. ALVO EM ESCOLAS PÚBLICAS vs PRIVADAS
# ============================================================
secao(8, 'ALVO POR REDE (PÚBLICA vs PRIVADA)')
if 'REDE_PUBLICA' in df.columns:
    for alvo in ALVOS:
        sub(f'{alvo} — por REDE_PUBLICA')
        tab = df.groupby('REDE_PUBLICA')[alvo].agg(
            n='count', media='mean', mediana='median', max='max'
        ).round(3)
        print(tab.to_string())

# ============================================================
# 9. ALVO POR LOCALIZAÇÃO E DEPENDÊNCIA
# ============================================================
secao(9, 'ALVO POR LOCALIZACAO / DEPENDENCIA')
for cat in ['LOCALIZACAO', 'DEPENDENCIA']:
    if cat in df.columns:
        sub(f'{cat} × alvos (média)')
        tab = df.groupby(cat)[ALVOS].agg(['count', 'mean']).round(3)
        print(tab.to_string())

# ============================================================
# 10. CORRELAÇÃO DAS NUMÉRICAS COM OS ALVOS
# ============================================================
secao(10, 'CORRELAÇÃO COM OS ALVOS')
# Pearson — só numéricas, ignora nulos par a par
corr = {}
for alvo in ALVOS:
    corr[alvo] = df[num_cols].corrwith(df[alvo].astype('float64'), method='pearson')
corr_df = pd.DataFrame(corr).round(4)
# Ordena pelo alvo principal (FUN_AF)
if 'ABANDONO_FUN_AF_T1' in corr_df.columns:
    corr_df = corr_df.reindex(corr_df['ABANDONO_FUN_AF_T1'].abs().sort_values(ascending=False).index)
print(corr_df.to_string())

# ============================================================
# 11. DETECÇÃO DE POSSÍVEL LEAKAGE
# ============================================================
secao(11, 'POSSÍVEL LEAKAGE')
# Alvos "irmãos": mesma variável derivada, mesmo período
irmaos = [a for a in ALVOS]
sub('Alvos irmãos (mesmo período, variável derivada)')
for a in irmaos:
    print(f'  {a} — se o alvo for outro, remover')

sub('Colunas derivadas do próprio alvo (mesmo ano, mesmo segmento)')
sugestoes_leak = []
for c in num_cols:
    # mesmo segmento do alvo, sem sufixo de tempo
    for alvo in ALVOS:
        if c != alvo and c.replace('_T1', '') in alvo.replace('_T1', ''):
            sugestoes_leak.append((c, alvo, 'mesmo segmento, mesmo período'))
print('Detecção automática:', sugestoes_leak if sugestoes_leak else 'nenhuma óbvia')

sub('Colunas com correlação > 0.95 com algum alvo')
for alvo in ALVOS:
    altas = corr_df[alvo].dropna().abs()
    altas = altas[altas > 0.95].index.tolist()
    print(f'  {alvo}: {altas}')

# ============================================================
# 12. DUPLICATAS
# ============================================================
secao(12, 'DUPLICATAS')
dup_full = df.duplicated().sum()
print(f'Linhas completamente duplicadas: {dup_full}')
dup_key = df.duplicated(subset=['CO_ENTIDADE', 'ANO']).sum()
print(f'Duplicatas por (CO_ENTIDADE, ANO): {dup_key}')
if dup_key > 0:
    dups = df[df.duplicated(subset=['CO_ENTIDADE', 'ANO'], keep=False)]
    print(dups[['CO_ENTIDADE', 'NO_ENTIDADE', 'ANO', 'ANO_ALVO']].head(20).to_string(index=False))

# ============================================================
# 13. OUTLIERS (IQR) POR COLUNA NUMÉRICA
# ============================================================
secao(13, 'OUTLIERS (IQR × 3)')
outliers = []
for c in num_cols:
    s = df[c].dropna().astype('float64')
    if len(s) == 0:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
    n_out = ((s < lo) | (s > hi)).sum()
    if n_out > 0:
        outliers.append((c, n_out, round(100 * n_out / len(s), 2), round(lo, 2), round(hi, 2)))
outliers.sort(key=lambda t: -t[1])
print(f'{"coluna":<28} {"n_out":>7} {"%":>7} {"lo":>10} {"hi":>10}')
for c, n, p, lo, hi in outliers[:40]:
    print(f'{c:<28} {n:>7} {p:>6.2f}% {lo:>10} {hi:>10}')

# ============================================================
# 14. COBERTURA DE VARIÁVEIS CRÍTICAS POR ANO_ALVO
# ============================================================
secao(14, 'COBERTURA DE VARIÁVEIS CRÍTICAS POR ANO_ALVO')
criticas = [
    'ATU_FUN_AF', 'TDI_FUN_AF', 'IED_FUN_ALTO',
    'APROVACAO_FUN_AF', 'REPROVACAO_FUN_AF', 'ABANDONO_FUN_AF',
    'ATU_MED_TOTAL', 'TDI_MED_TOTAL', 'IED_MED_ALTO',
    'APROVACAO_MED', 'REPROVACAO_MED', 'ABANDONO_MED',
]
criticas = [c for c in criticas if c in df.columns]
cob = df.groupby('ANO_ALVO')[criticas].apply(lambda g: g.notna().mean().round(3))
print(cob.to_string())

# ============================================================
# 15. DISTRIBUIÇÃO TEMPORAL DO ALVO PRINCIPAL
# ============================================================
secao(15, 'EVOLUÇÃO TEMPORAL DO ALVO PRINCIPAL (ABANDONO_FUN_AF_T1)')
alvo_p = 'ABANDONO_FUN_AF_T1'
if alvo_p in df.columns:
    tab = df.groupby('ANO_ALVO')[alvo_p].agg(
        n='count', media='mean', mediana='median',
        p10=lambda s: s.quantile(0.10),
        p90=lambda s: s.quantile(0.90),
        max='max'
    ).round(3)
    print(tab.to_string())

# ============================================================
# 16. ESCOLAS QUE APARECEM EM TODOS OS ANOS
# ============================================================
secao(16, 'PAINEL BALANCEADO?')
n_por_escola = df.groupby('CO_ENTIDADE')['ANO'].nunique()
print('Escolas por nº de anos distintos que aparecem:')
print(n_por_escola.value_counts().sort_index().to_string())
print(f'\nTotal de escolas únicas: {df["CO_ENTIDADE"].nunique():,}')
print(f'Escolas em todos os 7 anos: {(n_por_escola == 7).sum():,}')

# ============================================================
# 17. RESUMO E RECOMENDAÇÕES
# ============================================================
secao(17, 'RESUMO ESTRUTURADO (para IA)')
print(f'''
CONTEXTO
- Unidade         : escola-ano
- Período         : ANO {df["ANO"].min()}–{df["ANO"].max()} → alvo em ANO_ALVO {df["ANO_ALVO"].min()}–{df["ANO_ALVO"].max()}
- Escolas únicas  : {df["CO_ENTIDADE"].nunique()}
- Linhas          : {len(df)}

ALVOS DISPONÍVEIS
- ABANDONO_FUN_T1     : n={df["ABANDONO_FUN_T1"].notna().sum()}, média={df["ABANDONO_FUN_T1"].mean():.3f}
- ABANDONO_FUN_AF_T1  : n={df["ABANDONO_FUN_AF_T1"].notna().sum()}, média={df["ABANDONO_FUN_AF_T1"].mean():.3f}   ← alvo principal
- ABANDONO_MED_T1     : n={df["ABANDONO_MED_T1"].notna().sum()}, média={df["ABANDONO_MED_T1"].mean():.3f}

PROBLEMAS IDENTIFICADOS
1. ANO_ALVO ausente em {df["ANO_ALVO"].isna().sum()} linhas ({df["ANO_ALVO"].isna().mean()*100:.1f}%) → último ano da série, sem alvo.
2. Missing estrutural: variáveis de um segmento só existem em escolas que oferecem aquele segmento.
3. Colunas totalmente constantes: {constantes}
4. Duplicatas por (CO_ENTIDADE, ANO): {dup_key}

RECOMENDAÇÕES PARA O PRÉ-PROCESSAMENTO
- Filtrar REDE_PUBLICA == 1 antes de modelar (o modelo é sobre rede pública).
- Separar em dois modelos: anos finais (alvo ABANDONO_FUN_AF_T1) e médio (alvo ABANDONO_MED_T1).
- Remover de features os alvos irmãos (não usar ABANDONO_FUN_T1 nem ABANDONO_MED_T1 quando o alvo for ABANDONO_FUN_AF_T1, e vice-versa).
- Imputação: mediana para numéricas, 'Missing' para categóricas.
- Encoding: OneHot para LOCALIZACAO, DEPENDENCIA, CO_MUNICIPIO.
- Validação temporal: treino ANO ≤ 2022, validação 2023, teste 2024→2025.
- Escalar numéricas: SIM (Ridge precisa; árvores não se importam).
''')


  1. FORMA E TIPOS
Linhas  : 6,054
Colunas : 59

Distribuição de dtypes:
Float64           46
string[python]     6
Int8               5
Int16              2

--- Colunas numéricas (53) ---
['ANO', 'ANO_ALVO', 'REDE_PUBLICA', 'DISP_ATU', 'DISP_TDI', 'DISP_IED', 'DISP_REND', 'ATU_FUN_TOTAL', 'ATU_FUN_AI', 'ATU_FUN_AF', 'ATU_MED_TOTAL', 'TDI_FUN_TOTAL', 'TDI_FUN_AI', 'TDI_FUN_AF', 'TDI_MED_TOTAL', 'IED_FUN_N1', 'IED_FUN_N2', 'IED_FUN_N3', 'IED_FUN_N4', 'IED_FUN_N5', 'IED_FUN_N6', 'IED_FUN_ALTO', 'IED_MED_N1', 'IED_MED_N2', 'IED_MED_N3', 'IED_MED_N4', 'IED_MED_N5', 'IED_MED_N6', 'IED_MED_ALTO', 'APROVACAO_FUN', 'APROVACAO_FUN_AI', 'APROVACAO_FUN_AF', 'APROVACAO_MED', 'REPROVACAO_FUN', 'REPROVACAO_FUN_AI', 'REPROVACAO_FUN_AF', 'REPROVACAO_MED', 'ABANDONO_FUN', 'ABANDONO_FUN_AI', 'ABANDONO_FUN_AF', 'ABANDONO_MED', 'ABANDONO_FUN_LAG1', 'ABANDONO_FUN_AF_LAG1', 'ABANDONO_MED_LAG1', 'TDI_FUN_AF_DELTA1', 'TDI_MED_TOTAL_DELTA1', 'ATU_FUN_AF_DELTA1', 'ATU_MED_TOTAL_DELTA1', 'ABANDONO_FUN_AF_DELTA1

c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\gusta\Documents\Projeto da Gabi\teste censo completo\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
